# Environment & Git Setup Notes

Personal reference for how this project's conda environment and GitHub repo were set up, plus the activation fix needed for PowerShell. Commands are PowerShell / Git commands, not Python — copy-paste into a terminal, don't run these cells.

## 1. Create the environment in VS Code

Despite the folder being named `venv`, this was created as a **conda environment** (via `-p`, i.e. `--prefix`), not a standard Python `venv`. That distinction matters later for activation.

```powershell
conda create -p venv python==3.8 -y
conda activate venv/
```

## 2. Initialize git and make the first commit

```powershell
git init
git add README.md
git commit -m "first commit"
git status
```

## 3. Rename branch and connect to GitHub

```powershell
git branch -M main
git remote add origin https://github.com/Satya666-pra/MLPROJECT.git
git remote -v
git config --global user.email
git push -u origin main
```

## 4. Sync remote changes

```powershell
git pull
```

## 5. Gotcha: `pip install` fails with "externally-managed-environment"

This happened when `pip install -r requirements.txt` was run without the environment properly activated in that terminal — pip fell back to the protected base/system Python instead of the project's env.

**Root cause found:** `conda activate` alone did nothing in a plain PowerShell session because conda's shell hook wasn't initialized there yet (`conda` resolved to the raw `.exe`, not the shell-integrated function). So `$env:CONDA_PREFIX` never got set, and `python`/`pip` kept pointing outside the env.

**One-time permanent fix:**
```powershell
conda init powershell
```
Then **close and reopen the terminal**. After that, plain `conda activate` works in every new PowerShell window.

**If you ever see the hook missing again** (e.g. a terminal opened in a weird way), load it manually for that one session:
```powershell
(& "C:\Users\satya\miniconda3-pbe\Scripts\conda.exe" "shell.powershell" "hook") | Out-String | Invoke-Expression
conda activate ".\venv"
```

## 6. Correct day-to-day workflow (after the one-time fix above)

```powershell
conda activate ".\venv"
pip install -r requirements.txt
```

**How to verify the env is actually active before installing anything:**
```powershell
$env:CONDA_PREFIX        # should show the path ending in \venv
python --version         # should show Python 3.8.0
pip --version             # path shown should be inside \venv\lib\site-packages\pip
```

## 7. What `setup.py` does and why it exists

`setup.py` is what turns this folder into an installable Python **package** (named `mlproject`) instead of just a loose collection of scripts. It's what runs behind the scenes when `pip install -e .` is executed.

```python
from setuptools import find_packages, setup
from typing import List

HYPEN_E_DOT = '-e .'
def get_requirements(file_path: str) -> List[str]:
    '''
    this function will return the list of requirements
    '''
    requirements = []
    with open(file_path) as file_obj:
        requirements = file_obj.readlines()
        requirements = [req.replace("\n", "") for req in requirements]

        if HYPEN_E_DOT in requirements:
            requirements.remove(HYPEN_E_DOT)

    return requirements

setup(
    name='mlproject',
    version='0.0.1',
    author='SatyaprakashKabi',
    author_email='satyaprakashKabi@gmail.com',
    packages=find_packages(),
    install_requires=get_requirements('requirements.txt')
)
```

**Line-by-line:**

- `from setuptools import find_packages, setup` — `setup()` is the function that actually registers the package's metadata with pip/setuptools. `find_packages()` auto-discovers every directory containing an `__init__.py` (i.e. every importable sub-package) so they don't have to be listed by hand.

- `HYPEN_E_DOT = '-e .'` — a constant holding the literal text `-e .`, the line that appears at the bottom of `requirements.txt`. `-e .` tells pip "install *this* project itself, in editable mode" (see below). It's a pip-only instruction, not a real package name, so it must never be passed to `install_requires`.

- `get_requirements(file_path)` — reads `requirements.txt` and turns it into the list of dependency strings `setup()` needs:
  - `file_obj.readlines()` reads every line, including its trailing `\n`.
  - the list comprehension strips those `\n` characters off each line.
  - the `if HYPEN_E_DOT in requirements` check removes the `-e .` entry if present — **this is the important guard**. Without it, `setup()` would try to treat `-e .` as a package name and installation would break, because `-e .` only makes sense as a pip command-line flag, not as an item in `install_requires`.

- `setup(...)` — the actual package registration:
  - `name='mlproject'` — the package's importable/installed name.
  - `version='0.0.1'` — package version.
  - `author`, `author_email` — metadata only, no functional effect.
  - `packages=find_packages()` — the list of sub-packages to include (auto-detected).
  - `install_requires=get_requirements('requirements.txt')` — the dependency list, reusing `requirements.txt` as the single source of truth instead of duplicating package names in two places.

**How this connects back to `requirements.txt` and the earlier `pip install` step:**

`requirements.txt` ends with `-e .`. That line is what made `pip install -r requirements.txt` (run in Section 6) also install *this project itself* as an editable package — which is why the install log showed `Running setup.py develop for mlproject` and `Successfully installed ... mlproject`. "Editable" means the package is linked back to this source folder rather than copied into `site-packages`, so any code changes here are picked up immediately without reinstalling — useful during active development, and why `from src...` / `import mlproject...`-style imports work project-wide once the env is activated.

## 8. Gotcha: `src/__init__.py` missing from `SOURCES.txt`

`mlproject.egg-info/SOURCES.txt` is auto-generated every time `pip install -e .` runs, listing whatever `find_packages()` discovers at that moment. At one point it showed `src/__init__.py` on line 8, but the file on disk had disappeared — it had been created locally at some earlier point, never committed to git, and then got deleted, so the next reinstall regenerated `SOURCES.txt` without it.

**Fix — recreate the package folder and reinstall:**
```powershell
mkdir src
New-Item src\__init__.py -ItemType File

conda activate ".\venv"
pip install -e . --no-deps

Get-Content mlproject.egg-info\SOURCES.txt   # confirm src/__init__.py is listed again
```
`--no-deps` skips reinstalling pandas/numpy/etc. — it just re-registers the local package so `find_packages()` picks up the new `src/` folder.

## 9. Gotcha: `.gitignore` rules weren't being applied

`venv/` and `mlproject.egg-info/` kept showing up as untracked in `git status`, even though the ignore file already listed `venv/` and `*.egg-info/`. The cause: the file was named **`gitignore`** (no leading dot) instead of **`.gitignore`** — Git only recognizes the dotted filename, so the rules were silently never applied.

**Fix — rename it and re-check status:**
```powershell
git mv gitignore .gitignore
git status
```
If `git mv` errors with `fatal: bad source` (can happen on Windows), do it as a plain rename plus a manual stage instead — git still detects it as a rename once both paths are staged together:
```powershell
Rename-Item gitignore .gitignore
git add gitignore .gitignore
git status   # should show "renamed: gitignore -> .gitignore"
```

## 10. Committing and pushing only the real source files

Never commit `venv/` (the local environment) or `mlproject.egg-info/` (generated build metadata) — both are covered by `.gitignore` once Section 9's fix is applied. Stage only the actual project files:

```powershell
git add .gitignore README.md environment_setup_notes.ipynb requirements.txt setup.py src/
git status                          # sanity check — venv/ and egg-info/ should NOT appear
git commit -m "Add src package, setup.py, requirements, and rename gitignore to .gitignore"
git push
```

## 11. Gotcha: `python src\exception.py` works from VS Code's Run button but not from the terminal

- **Run ▶ button works** because VS Code invokes the *selected interpreter's* `python.exe` (`venv\python.exe`) directly, bypassing the shell entirely. That interpreter already has `mlproject` installed (editable install from Section 7), so `import src...` succeeds.
- **Terminal fails** with `ModuleNotFoundError: No module named 'src'` because a fresh PowerShell terminal auto-activates conda's **`base`** environment (default `conda init` behavior), not this project's `venv`. Base has a different Python with no `mlproject` installed.

**Fix — always activate the project env explicitly before running scripts in a terminal:**
```powershell
conda activate .\venv
python src\exception.py
```
Don't rely on VS Code's auto-activation settings (`python.terminal.activateEnvironment`, the newer `python-envs.*` settings) to always pick the right env automatically — they can conflict with each other. Explicitly running `conda activate .\venv` in the terminal is the reliable habit.

**Also fixed:** pinned `.vscode/settings.json` so the workspace defaults to `venv\python.exe`:
```json
{
    "python.defaultInterpreterPath": "${workspaceFolder}\\venv\\python.exe",
    "python.terminal.activateEnvironment": true
}
```
If this default doesn't seem to apply, manually run **Ctrl+Shift+P → "Python: Select Interpreter"** and pick `venv\python.exe` — VS Code only auto-applies the default when no interpreter has been manually chosen before for this workspace.

## 12. Deep dive: `src/logger.py`

```python
import logging
import os
from datetime import datetime

LOG_FILE = f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"
logs_path = os.path.join(os.getcwd(), "logs")
os.makedirs(logs_path, exist_ok=True)

LOG_FILE_PATH = os.path.join(logs_path, LOG_FILE)

logging.basicConfig(
    filename=LOG_FILE_PATH,
    format="[ %(asctime)s ] %(lineno)d %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)
```

**Purpose:** a shared, pre-configured logger that every other module in `src/` can import, so log formatting and destination are defined in exactly one place.

**Line-by-line:**

- `LOG_FILE = f"{datetime.now().strftime('%m_%d_%Y_%H_%M_%S')}.log"` — builds a filename from the current timestamp (e.g. `09_26_2026_17_01_51.log`), so **every run of the program gets its own fresh log file** rather than all runs appending to one shared file.

- `logs_path = os.path.join(os.getcwd(), "logs")` — the target directory is `logs/` under the **current working directory** (`os.getcwd()`), not necessarily the project root. This matters: if you run a script from a different folder, `os.getcwd()` changes, and a *new* `logs/` folder appears wherever you happened to run the command from. (This is why you may have noticed log folders showing up in unexpected places if scripts were run from different directories.)

- `os.makedirs(logs_path, exist_ok=True)` — creates the `logs/` folder if it doesn't already exist. `exist_ok=True` prevents an error on subsequent runs when the folder is already there.

- `LOG_FILE_PATH = os.path.join(logs_path, LOG_FILE)` — the full path to this run's log file, e.g. `.../logs/09_26_2026_17_01_51.log`.
  - **Bug that was fixed here (see Section 13-equivalent earlier troubleshooting):** the original code passed `LOG_FILE` (already ending in `.log`) into `os.path.join(os.getcwd(), "logs", LOG_FILE)` and then called `os.makedirs()` on *that full file path* — which created a **directory** named like a log file (e.g. `logs/09_26_2026_16_58_55.log/`), and then wrote the actual log file *inside* that oddly-named folder. The fix was to make `logs_path` just the `logs` directory, and only join in `LOG_FILE` afterward for the actual file path.

- `logging.basicConfig(...)` — configures Python's built-in **root logger** (the shared, global logger object) exactly once for the whole process:
  - `filename=LOG_FILE_PATH` — sends log output to this file instead of the console.
  - `format="[ %(asctime)s ] %(lineno)d %(name)s - %(levelname)s - %(message)s"` — defines what each log line looks like: timestamp, the line number the log call was made from, the logger's name (`root` by default here), the severity level (INFO/WARNING/ERROR/...), and the actual message.
  - `level=logging.INFO` — the minimum severity that gets recorded. `DEBUG` messages would be *silently ignored*; `INFO`, `WARNING`, `ERROR`, and `CRITICAL` all get written.

**Important gotcha with `basicConfig`:** it only has an effect the **first time** it's called in a given Python process. Since this configuration runs automatically the moment `logger.py` is imported (there's no `if __name__ == "__main__"` guard around it), the very first `from src.logger import logging` anywhere in the program sets up the file/format/level for the *entire run* — every other module that later does `from src.logger import logging` just reuses that same already-configured root logger.

## 13. Deep dive: `src/exception.py`

```python
import sys
from src.logger import logging

def error_message_detail(error, error_detail: sys):
    _, _, exc_tb = error_detail.exc_info()
    file_name = exc_tb.tb_frame.f_code.co_filename
    error_message = "Error occured in python script name [{0}] line number [{1}] error message[{2}]".format(
        file_name, exc_tb.tb_lineno, str(error))
    return error_message

class CustomException(Exception):
    def __init__(self, error_message, error_detail: sys):
        super().__init__(error_message)
        self.error_message = error_message_detail(error_message, error_detail=error_detail)

    def __str__(self):
        return self.error_message

if __name__ == "__main__":
    try:
        a = 1 / 0
    except Exception as e:
        logging.info(error_message_detail(e, sys))
        raise CustomException(e, sys)
```

**Purpose:** a single, project-wide custom exception class. The idea (standard in this ML project template) is that *every* `except` block anywhere in the codebase re-raises the caught error as `CustomException(e, sys)`, so no matter where something breaks, the final error message always tells you the exact file, the exact line, and the original error text — instead of a bare, generic traceback.

**Line-by-line:**

- `import sys` — needed for `sys.exc_info()`, Python's way of asking "what exception is currently being handled?"

- `from src.logger import logging` — importing this line is what actually **triggers** all of `logger.py`'s module-level code (Section 12) the first time it runs, wiring up the log file destination/format for the whole program.

- `error_message_detail(error, error_detail: sys)` — builds the descriptive message string:
  - `error_detail.exc_info()` — here `error_detail` is the `sys` module itself (passed in by the caller as `sys`), and `sys.exc_info()` returns a 3-tuple `(exception_type, exception_value, traceback_object)` describing whatever exception is currently being handled.
  - `_, _, exc_tb = ...` — the type and value are discarded (`_`); only the **traceback object** (`exc_tb`) is kept, because that's what holds the file/line info.
  - `exc_tb.tb_frame.f_code.co_filename` — drills into the traceback: `tb_frame` is the stack frame where the exception occurred, `f_code` is that frame's compiled code object, and `co_filename` is the source filename it belongs to.
  - `exc_tb.tb_lineno` — the exact line number in that file where the exception was raised.
  - the `.format(...)` call assembles all three pieces — filename, line number, and `str(error)` (the original exception's message) — into one human-readable string.

- `class CustomException(Exception)` — a custom exception type that wraps the original one with extra context:
  - `super().__init__(error_message)` — calls the base `Exception.__init__`, storing whatever was passed as `error_message` (note: in this codebase's usage, that's actually the *original exception object* `e`, not a string — slightly confusing naming, but harmless since it's immediately overwritten below).
  - `self.error_message = error_message_detail(error_message, error_detail=error_detail)` — immediately replaces that with the fully detailed string built above, and stores it as an instance attribute.
  - `def __str__(self): return self.error_message` — overrides how the exception renders as text. This is why, when `CustomException` is left uncaught, Python's default traceback printer (which calls `str()` on the exception) shows the **detailed** message rather than Python's default repr of the wrapped exception.

- `if __name__ == "__main__": ...` — a self-test block that **only runs when this file is executed directly** (`python src\exception.py`), not when it's imported by another module elsewhere in the project:
  - `a = 1 / 0` — deliberately triggers a `ZeroDivisionError` to have something to catch.
  - `logging.info(error_message_detail(e, sys))` — writes the same detailed message into the log file (Section 12's logger). This was a deliberate fix — the original code only logged the generic string `"Divide by zero error"`, which wasn't useful for debugging later since it lacked file/line/error detail.
  - `raise CustomException(e, sys)` — wraps the caught `ZeroDivisionError` and re-raises it. Since nothing catches it further up, Python prints the full chained traceback (`ZeroDivisionError` → "During handling of the above exception, another exception occurred" → `CustomException: Error occured in python script name [...] line number [...] error message[...]`).


## 14. Deep dive: `src/components/data_ingestion.py`

```python
import os
import sys
from src.exception import CustomException
from src.logger import logging
import pandas as pd

from sklearn.model_selection import train_test_split
from dataclasses import dataclass

from src.components.data_transformation import DataTransformation
from src.components.data_transformation import DataTransformationConfig

from src.components.model_trainer import ModelTrainerConfig
from src.components.model_trainer import ModelTrainer

@dataclass
class DataIngestionConfig:
    train_data_path: str = os.path.join('artifacts', "train.csv")
    test_data_path: str = os.path.join('artifacts', "test.csv")
    raw_data_path: str = os.path.join('artifacts', "data.csv")

class DataIngestion:
    def __init__(self):
        self.ingestion_config = DataIngestionConfig()

    def initiate_data_ingestion(self):
        logging.info("Entered the data ingestion method or component")
        try:
            df = pd.read_csv('notebook\\data\\stud.csv')
            logging.info('Read the dataset as dataframe')

            os.makedirs(os.path.dirname(self.ingestion_config.train_data_path), exist_ok=True)

            df.to_csv(self.ingestion_config.raw_data_path, index=False, header=True)

            logging.info("Train test split initiated")
            train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

            train_set.to_csv(self.ingestion_config.train_data_path, index=False, header=True)
            test_set.to_csv(self.ingestion_config.test_data_path, index=False, header=True)

            logging.info("Ingestion of the data is completed")

            return (
                self.ingestion_config.train_data_path,
                self.ingestion_config.test_data_path
            )
        except Exception as e:
            raise CustomException(e, sys)

if __name__ == "__main__":
    obj = DataIngestion()
    train_data, test_data = obj.initiate_data_ingestion()

    data_transformation = DataTransformation()
    train_arr, test_arr, _ = data_transformation.initiate_data_transformation(train_data, test_data)

    modeltrainer = ModelTrainer()
    print(modeltrainer.initiate_model_trainer(train_arr, test_arr))
```

**Purpose:** this is the first stage of the production pipeline (as opposed to the exploratory notebooks) — it reads the raw dataset, saves a copy of it, splits it into train/test sets, saves those too, and hands off *file paths* (not DataFrames) to the next stage. It's the script equivalent of what `1. EDA` and `2. MODEL TRAINING` do interactively, but structured to be chainable and reusable.

**Line-by-line:**

- `from src.exception import CustomException` / `from src.logger import logging` — reuses the shared exception/logging infrastructure from Sections 12–13, so every step here logs consistently and any failure is wrapped with full file/line detail.

- `from src.components.data_transformation import DataTransformation, DataTransformationConfig` and `from src.components.model_trainer import ModelTrainerConfig, ModelTrainer` — pulls in the *next two stages* of the pipeline. Even though this file is called "data_ingestion.py," it imports the transformation and training stages too, because its `if __name__ == "__main__":` block (see below) chains all three together into one runnable end-to-end pipeline.

- `@dataclass class DataIngestionConfig` — `@dataclass` is a decorator that auto-generates boilerplate (`__init__`, `__repr__`, `__eq__`) for a class that's mainly just a bag of fields. Here it defines three file paths as class attributes with defaults:
  - `train_data_path = 'artifacts/train.csv'`
  - `test_data_path = 'artifacts/test.csv'`
  - `raw_data_path = 'artifacts/data.csv'`
  Centralizing these in one config object (instead of hardcoding path strings throughout the class) means all ingestion output locations are defined in exactly one place.

- `class DataIngestion: def __init__(self): self.ingestion_config = DataIngestionConfig()` — creates one instance of the config above and stores it, so `initiate_data_ingestion` can reference `self.ingestion_config.train_data_path` etc.

- `initiate_data_ingestion(self)` — the actual pipeline step, wrapped in the project's standard `try/except -> CustomException` pattern:
  - `df = pd.read_csv('notebook\\data\\stud.csv')` — loads the raw dataset. **Gotcha, same class of issue as Section 12's `logger.py`:** this is a relative path, so it only works if the script is run with the project root as the current working directory (`os.getcwd()`). Run it from anywhere else (e.g. `cd src\\components` first) and this fails with `FileNotFoundError`. It also isn't a raw string (`r'...'`) or `os.path.join(...)`-built path — it happens to work here because `\\d` isn't a recognized Python escape sequence, but relying on that is fragile.
  - `os.makedirs(os.path.dirname(self.ingestion_config.train_data_path), exist_ok=True)` — `os.path.dirname('artifacts/train.csv')` evaluates to `'artifacts'`, so this creates the `artifacts/` folder if it doesn't exist yet. `exist_ok=True` avoids an error on repeated runs.
  - `df.to_csv(self.ingestion_config.raw_data_path, ...)` — saves the full, unsplit dataset to `artifacts/data.csv` as a reference copy of exactly what was ingested.
  - `train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)` — the same 80/20 split used in the `2. MODEL TRAINING` notebook, with a fixed `random_state` for reproducibility.
  - Both splits get saved to their own CSVs (`artifacts/train.csv`, `artifacts/test.csv`).
  - `return (self.ingestion_config.train_data_path, self.ingestion_config.test_data_path)` — **importantly, this returns file *paths*, not the DataFrames themselves.** This is a deliberate pipeline design pattern: each stage persists its output to disk and only passes a *reference* (the path) to the next stage, rather than passing Python objects directly in memory. That means each stage could be run independently, resumed without rerunning earlier stages, or run as a genuinely separate process — much closer to how a real production pipeline (e.g. Airflow, a scheduled job) would be structured, versus the notebooks where everything lives in one interactive session's memory.
  - `except Exception as e: raise CustomException(e, sys)` — any failure anywhere in the `try` block (bad CSV path, disk write failure, etc.) gets wrapped in the project's `CustomException`, so the resulting traceback always shows the exact file/line/error rather than a bare exception.

- `if __name__ == "__main__": ...` — only runs when this file is executed directly (`python src\\components\\data_ingestion.py`), not when it's imported elsewhere. It chains all three pipeline stages together end-to-end:
  1. `obj = DataIngestion(); train_data, test_data = obj.initiate_data_ingestion()` — runs ingestion, gets back the train/test **CSV paths**.
  2. `data_transformation = DataTransformation(); train_arr, test_arr, _ = data_transformation.initiate_data_transformation(train_data, test_data)` — passes those paths into the transformation stage, which presumably re-reads the CSVs from disk, applies preprocessing (encoding/scaling, similar to the `ColumnTransformer` built in the `2. MODEL TRAINING` notebook), and returns transformed train/test arrays plus a third value (commonly a path to a saved preprocessor object in this project template — discarded here with `_` since it isn't needed at this point).
  3. `modeltrainer = ModelTrainer(); print(modeltrainer.initiate_model_trainer(train_arr, test_arr))` — feeds the transformed arrays into the training stage, which presumably fits/compares candidate models (mirroring the notebook's model-comparison loop) and returns/prints the best result.

**Minor cosmetic note:** the original log messages had typos (`"Inmgestion of the data iss completed"`) — harmless, but corrected above for clarity.


## 15. Gotcha: `ImportError: cannot import name 'DataTransformation'` when running `data_ingestion.py`

```
Traceback (most recent call last):
  File "...src/components/data_ingestion.py", line 10, in <module>
    from src.components.data_transformation import DataTransformation
ImportError: cannot import name 'DataTransformation' from 'src.components.data_transformation'
```

**Root cause:** `src/components/data_transformation.py` and `src/components/model_trainer.py` were both **completely empty files** at this point in the course — they exist as placeholders in the project structure but hadn't been written yet. `data_ingestion.py` imports `DataTransformation`/`DataTransformationConfig` and `ModelTrainerConfig`/`ModelTrainer` from them at the top of the file (module-level imports), and **Python executes all imports before anything else runs** — before even the `if __name__ == "__main__":` block. So the script crashed immediately at import time; none of the actual ingestion logic (`initiate_data_ingestion()`) had run yet at all.

**How this was isolated:** rather than assuming the ingestion code itself was broken, the imports and the downstream `data_transformation`/`model_trainer` calls in the `__main__` block were temporarily commented out (with `# TODO: re-enable once data_transformation.py / model_trainer.py are written` markers) so `initiate_data_ingestion()` could be tested standalone:

```python
# TODO: re-enable once data_transformation.py / model_trainer.py are written
# from src.components.data_transformation import DataTransformation
# from src.components.data_transformation import DataTransformationConfig

# from src.components.model_trainer import ModelTrainerConfig
# from src.components.model_trainer import ModelTrainer
...
if __name__=="__main__":
    obj=DataIngestion()
    train_data,test_data=obj.initiate_data_ingestion()
    print("Data ingestion completed. Train path:", train_data, "| Test path:", test_data)

    # TODO: re-enable once data_transformation.py / model_trainer.py are written
    # data_transformation=DataTransformation()
    # train_arr,test_arr,_=data_transformation.initiate_data_transformation(train_data,test_data)

    # modeltrainer=ModelTrainer()
    # print(modeltrainer.initiate_model_trainer(train_arr,test_arr))
```

**Result — confirmed the ingestion code itself is correct.** Running it standalone:
```powershell
conda activate ".\venv"
python src\components\data_ingestion.py
```
produced exactly the expected output:
- `artifacts/data.csv` — 1000 rows (full raw dataset + header)
- `artifacts/train.csv` — 800 rows (80% split)
- `artifacts/test.csv` — 200 rows (20% split)

matching the `test_size=0.2` split and preserving the original columns/header. The `ImportError` was purely a consequence of the pipeline's later stages not existing yet, not a bug in `data_ingestion.py`.

**Next step:** once `data_transformation.py` and `model_trainer.py` are written (later in the course), uncomment the marked-out imports and `__main__` calls to re-chain the full ingestion → transformation → training pipeline.


## 16. Deep dive: `src/components/data_transformation.py`

```python
import sys
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.exception import CustomException
from src.logger import logging
import os

from src.utils import save_object

@dataclass
class DataTransformationConfig:
    preprocessor_obj_file_path = os.path.join('artifacts', "proprocessor.pkl")

class DataTransformation:
    def __init__(self):
        self.data_transformation_config = DataTransformationConfig()

    def get_data_transformer_object(self):
        try:
            numerical_columns = ["writing_score", "reading_score"]
            categorical_columns = [
                "gender", "race_ethnicity", "parental_level_of_education",
                "lunch", "test_preparation_course",
            ]

            num_pipeline = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ])

            cat_pipeline = Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("one_hot_encoder", OneHotEncoder()),
                ("scaler", StandardScaler(with_mean=False))
            ])

            logging.info(f"Categorical columns: {categorical_columns}")
            logging.info(f"Numerical columns: {numerical_columns}")

            preprocessor = ColumnTransformer([
                ("num_pipeline", num_pipeline, numerical_columns),
                ("cat_pipelines", cat_pipeline, categorical_columns)
            ])

            return preprocessor
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_data_transformation(self, train_path, test_path):
        try:
            train_df = pd.read_csv(train_path)
            test_df = pd.read_csv(test_path)

            preprocessing_obj = self.get_data_transformer_object()

            target_column_name = "math_score"
            numerical_columns = ["writing_score", "reading_score"]

            input_feature_train_df = train_df.drop(columns=[target_column_name], axis=1)
            target_feature_train_df = train_df[target_column_name]

            input_feature_test_df = test_df.drop(columns=[target_column_name], axis=1)
            target_feature_test_df = test_df[target_column_name]

            input_feature_train_arr = preprocessing_obj.fit_transform(input_feature_train_df)
            input_feature_test_arr = preprocessing_obj.transform(input_feature_test_df)

            train_arr = np.c_[input_feature_train_arr, np.array(target_feature_train_df)]
            test_arr = np.c_[input_feature_test_arr, np.array(target_feature_test_df)]

            save_object(
                file_path=self.data_transformation_config.preprocessor_obj_file_path,
                obj=preprocessing_obj
            )

            return (
                train_arr,
                test_arr,
                self.data_transformation_config.preprocessor_obj_file_path,
            )
        except Exception as e:
            raise CustomException(e, sys)
```

**Purpose:** the second pipeline stage. It takes the train/test CSV **paths** produced by `data_ingestion.py` (Section 14), turns them into fully numeric, imputed, scaled arrays ready for model training, and — critically — saves the *fitted* preprocessing object to disk so the exact same transformation can be replayed later on brand-new data (e.g. in a Flask app at inference time), instead of accidentally re-fitting a fresh one on different data.

**Line-by-line:**

- Imports mirror the pattern from Sections 12–14: `CustomException`/`logging` for the shared error/log infrastructure, plus `ColumnTransformer`/`Pipeline`/`SimpleImputer`/`OneHotEncoder`/`StandardScaler` from sklearn for building the preprocessing steps.
- `from src.utils import save_object` — a project-level helper (presumably: pickle an arbitrary Python object to a given file path). **Heads-up:** `src/utils.py` is currently an **empty file** (0 bytes), same situation as Section 15's `data_transformation.py`/`model_trainer.py` gotcha. Running or importing this file right now will fail at this exact import line with `ImportError: cannot import name 'save_object' from 'src.utils'`, before any transformation logic runs — `save_object` needs to be written first.

- `@dataclass class DataTransformationConfig: preprocessor_obj_file_path = os.path.join('artifacts', "proprocessor.pkl")` — **subtle detail:** unlike `DataIngestionConfig`'s fields in Section 14 (`train_data_path: str = ...`), this attribute has **no type annotation** (`: str` is missing). In a `@dataclass`, only *annotated* class-level assignments become actual dataclass fields (get included in the auto-generated `__init__`/`__repr__`/etc.); an unannotated assignment like this is just a plain class attribute instead. It still works fine for this simple case (there's only one shared path, no need for per-instance overrides), but it means `@dataclass` isn't really doing anything functionally here. (Also note the typo: `"proprocessor.pkl"` instead of `"preprocessor.pkl"` — harmless, just the literal filename that will be created.)

- `get_data_transformer_object(self)` — builds and returns an **unfitted** `ColumnTransformer` (the actual fitting happens later, in `initiate_data_transformation`):
  - `numerical_columns = ["writing_score", "reading_score"]` / `categorical_columns = [...]` — same 2 numeric + 5 categorical columns identified in the `2. MODEL TRAINING` notebook, just hardcoded here instead of derived via `select_dtypes`.
  - `num_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])` — a `Pipeline` chains steps so the output of one feeds the next. For numeric columns: first fill any missing values with that column's **median** (robust to outliers, unlike mean), then standard-scale (zero mean, unit variance).
  - `cat_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("one_hot_encoder", OneHotEncoder()), ("scaler", StandardScaler(with_mean=False))])` — for categorical columns: fill missing values with the column's most common category, one-hot encode into binary indicator columns, then scale. `with_mean=False` is important here — one-hot encoded output is sparse (mostly zeros); centering it (subtracting the mean) would turn it into a dense matrix of small positive/negative floats, losing the memory benefits of sparsity and not making conceptual sense for binary indicator columns. Scaling by variance only (no centering) is the standard choice after one-hot encoding.
  - This is more production-robust than the notebook's simpler `ColumnTransformer`, which applied `OneHotEncoder`/`StandardScaler` directly with no missing-value handling at all.
  - `preprocessor = ColumnTransformer([("num_pipeline", num_pipeline, numerical_columns), ("cat_pipelines", cat_pipeline, categorical_columns)])` — combines both pipelines into one transformer that, given the full input DataFrame, routes each column subset to its own pipeline and concatenates the results.

- `initiate_data_transformation(self, train_path, test_path)` — the actual transformation step:
  - `train_df = pd.read_csv(train_path)` / `test_df = pd.read_csv(test_path)` — re-reads the CSVs from the paths handed over by `data_ingestion.py` (the file-path handoff pattern from Section 14).
  - `preprocessing_obj = self.get_data_transformer_object()` — builds the (still unfitted) `ColumnTransformer` above.
  - `input_feature_train_df = train_df.drop(columns=[target_column_name], axis=1)` / `target_feature_train_df = train_df[target_column_name]` — splits the training DataFrame into features (X) and target (y = `math_score`); same split done for the test DataFrame.
  - `input_feature_train_arr = preprocessing_obj.fit_transform(input_feature_train_df)` — **fits** the preprocessor (learns the median/most-frequent values, one-hot categories, and scaling parameters) using **only the training data**, and transforms it in the same call.
  - `input_feature_test_arr = preprocessing_obj.transform(input_feature_test_df)` — **only transforms** the test data, reusing the statistics already learned from training data — it deliberately does **not** call `fit_transform` again here. This is correct ML practice: fitting on test data (or on train+test combined) would leak information from the test set into the preprocessing step, inflating apparent model performance.
  - `train_arr = np.c_[input_feature_train_arr, np.array(target_feature_train_df)]` — `np.c_[...]` concatenates arrays **column-wise**. This appends the target column (`math_score`) as the **last column** of the transformed feature array, producing one combined array of shape `(n_samples, n_features + 1)` rather than returning `X`/`y` separately. Same construction for `test_arr`. This explains the return signature seen in `data_ingestion.py`'s `__main__` block (Section 14): `train_arr, test_arr, _ = data_transformation.initiate_data_transformation(...)` — `train_arr`/`test_arr` are single combined arrays with the target as the last column; a downstream `model_trainer.py` would be expected to slice it back apart (e.g. `X = train_arr[:, :-1]`, `y = train_arr[:, -1]`).
  - `save_object(file_path=self.data_transformation_config.preprocessor_obj_file_path, obj=preprocessing_obj)` — persists the **fitted** preprocessor (not the raw class, the actual object with learned parameters) to `artifacts/proprocessor.pkl`, presumably via `pickle`. This is what lets a later inference step (e.g. a Flask endpoint) load the exact same fitted encoder/scaler and apply it consistently to brand-new input, rather than needing the original training data again.
  - `return (train_arr, test_arr, self.data_transformation_config.preprocessor_obj_file_path)` — confirms the `_` discarded in `data_ingestion.py`'s `__main__` block (Section 14) is indeed this saved preprocessor's file path — not needed at that point in the chain, but available if something later wants to reload the preprocessor directly.
  - `except Exception as e: raise CustomException(e, sys)` — same shared error-wrapping pattern used throughout this project (Sections 12–15).

**Summary of what makes this "production-grade" versus the notebook version:** missing-value handling via `SimpleImputer` (the notebook's preprocessor had none), strict train-fit/test-transform-only discipline to avoid data leakage, and persisting the fitted preprocessor object to disk so training-time preprocessing can be exactly replayed at inference time.


## 17. Deep dive: `src/utils.py`

```python
import os
import sys

import numpy as np
import pandas as pd
import dill
import pickle
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV

from src.exception import CustomException

def save_object(file_path, obj):
    try:
        dir_path = os.path.dirname(file_path)
        os.makedirs(dir_path, exist_ok=True)

        with open(file_path, "wb") as file_obj:
            pickle.dump(obj, file_obj)

    except Exception as e:
        raise CustomException(e, sys)

def evaluate_models(X_train, y_train, X_test, y_test, models, param):
    try:
        report = {}

        for i in range(len(list(models))):
            model = list(models.values())[i]
            para = param[list(models.keys())[i]]

            gs = GridSearchCV(model, para, cv=3)
            gs.fit(X_train, y_train)

            model.set_params(**gs.best_params_)
            model.fit(X_train, y_train)

            y_train_pred = model.predict(X_train)
            y_test_pred = model.predict(X_test)

            train_model_score = r2_score(y_train, y_train_pred)
            test_model_score = r2_score(y_test, y_test_pred)

            report[list(models.keys())[i]] = test_model_score

        return report

    except Exception as e:
        raise CustomException(e, sys)

def load_object(file_path):
    try:
        with open(file_path, "rb") as file_obj:
            return pickle.load(file_obj)

    except Exception as e:
        raise CustomException(e, sys)
```

**Purpose:** a grab-bag of shared helper functions used across the pipeline's later stages — persisting/loading arbitrary Python objects to/from disk (used for the fitted preprocessor in Section 16, and would be used for the trained model too), and running hyperparameter-tuned model comparison (the piece `model_trainer.py`, per Section 15, still needs to be written to use).

**Imports — including a couple that go unused:**
- `numpy`/`pandas` are imported but **never actually referenced** anywhere in this file's three functions — harmless dead imports, likely leftover from copy-pasting boilerplate.
- `dill` is also imported but **unused** — only the standard library's `pickle` is actually called below. `dill` is a more permissive serialization library (can pickle things plain `pickle` can't, like lambdas or nested functions) that commonly shows up in this exact project template as an alternative to `pickle`, but in this version of the file, it was imported and then not wired in.
- `from src.exception import CustomException` — the same shared exception class from Section 13, used to wrap every failure here too.

**`save_object(file_path, obj)`:**
- `dir_path = os.path.dirname(file_path)` / `os.makedirs(dir_path, exist_ok=True)` — same defensive pattern seen in `data_ingestion.py` (Section 14): make sure the target directory exists before writing into it.
- `with open(file_path, "wb") as file_obj: pickle.dump(obj, file_obj)` — opens the file in **binary write mode** (`"wb"`, required because `pickle` serializes to raw bytes, not text) and serializes `obj` into it.
- This is exactly the function `data_transformation.py` imports and calls (Section 16: `save_object(file_path=..., obj=preprocessing_obj)`) to persist the *fitted* `ColumnTransformer` to `artifacts/proprocessor.pkl` — confirming the "presumably via pickle" note made there.
- `except Exception as e: raise CustomException(e, sys)` — the same shared error-wrapping pattern as every other file in `src/`.

**`evaluate_models(X_train, y_train, X_test, y_test, models, param)`:**
This is a hyperparameter-tuned upgrade of the plain model-comparison loop from the `2. MODEL TRAINING` notebook — instead of just fitting each candidate model with default settings, it tunes each one first, then reports test performance.
- `report = {}` — will end up as `{model_name: test_r2_score}` for every model, returned at the end.
- `for i in range(len(list(models))):` / `model = list(models.values())[i]` — same "iterate a dict by numeric index" style used in the notebook's model loop.
- `para = param[list(models.keys())[i]]` — looks up this model's specific hyperparameter search grid from a second dict, `param`, keyed by the same model names as `models` (e.g. `param["Random Forest Regressor"] = {"n_estimators": [...], "max_depth": [...]}`).
- `gs = GridSearchCV(model, para, cv=3)` — exhaustively tries every combination of hyperparameters in `para` for this model, using **3-fold cross-validation** to score each combination.
- `gs.fit(X_train, y_train)` — runs the grid search; internally this fits many versions of the model on rotating subsets of the training data to find `gs.best_params_`.
- `model.set_params(**gs.best_params_)` / `model.fit(X_train, y_train)` — takes the **original** model object and updates it with the best hyperparameters found, then refits it on the **entire** training set (not just the CV folds used internally by `GridSearchCV`). **Minor design note:** `GridSearchCV` already exposes `gs.best_estimator_`, which by default (`refit=True`) is already refit on the full training data with the best params — so this manual `set_params` + `fit` achieves the same end result as just using `gs.best_estimator_` directly, just via an extra explicit step rather than using what `GridSearchCV` already hands back.
- `y_train_pred = model.predict(X_train)` / `y_test_pred = model.predict(X_test)` — predictions on both sets.
- `train_model_score = r2_score(y_train, y_train_pred)` — **computed but never used again** — it's not stored in `report`, printed, or returned anywhere. This is dead computation, likely a leftover from adapting the notebook's `evaluate_model` helper (Section — see `2. MODEL TRAINING` notebook), which *did* use both train and test scores (to print both, for spotting overfitting); here only the test score ends up mattering.
- `test_model_score = r2_score(y_test, y_test_pred)` / `report[list(models.keys())[i]] = test_model_score` — this is the value that actually gets returned, per model.
- `return report` — a dict of `{model_name: test_r2}`, intended to be consumed by `model_trainer.py` (not yet written, per Section 15) to pick the best-performing model, mirroring the leaderboard step in the `2. MODEL TRAINING` notebook.
- `except Exception as e: raise CustomException(e, sys)` — same pattern.

**`load_object(file_path)`:**
- The inverse of `save_object`: opens the file in **binary read mode** (`"rb"`) and `pickle.load`s it back into a Python object. This is what a later inference step (e.g. a Flask endpoint) would use to reload the saved preprocessor (`artifacts/proprocessor.pkl`) and/or a saved trained model, without needing to refit anything from the original training data.
- `except Exception as e: raise CustomException(e, sys)` — same pattern.

**How this file fits into the pipeline so far:** `data_transformation.py` (Section 16) calls `save_object` to persist its fitted preprocessor; the not-yet-written `model_trainer.py` (Section 15) would be expected to call `evaluate_models` to tune/compare candidate models and `save_object` again to persist the winning trained model; and a future inference script (e.g. `predict_pipeline.py` or a Flask app) would call `load_object` twice — once for the preprocessor, once for the model — to serve predictions on new data using exactly what was learned during training.
